# Data Cleaning Challenge

## 1. Importing Libraries and Data Set

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv("dirty_cafe_sales.csv")
data

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


## 2. Standartized Column Names

In [4]:
data.columns = (
    data.columns
      .str.lower()
      .str.strip()
      .str.replace(r"\s+", "_", regex=True)
)

numeric_columns = [
    "price_per_unit",
    "quantity",
    "total_spent"
]

for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3.0,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3.0,NaN,3.0,Digital Wallet,NaN,2023-12-02


## 3. Filling Item Names Based on Their Prices

In [5]:
data[data["item"] == "UNKNOWN"]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
6,TXN_4433211,UNKNOWN,3.0,3.0,9.0,ERROR,Takeaway,2023-10-06
31,TXN_8927252,UNKNOWN,2.0,1.0,NaN,Credit Card,ERROR,2023-11-06
33,TXN_7710508,UNKNOWN,5.0,1.0,5.0,Cash,NaN,ERROR
36,TXN_6855453,UNKNOWN,4.0,3.0,12.0,NaN,In-store,2023-07-17
52,TXN_8914892,UNKNOWN,5.0,5.0,25.0,Digital Wallet,NaN,2023-03-15
...,...,...,...,...,...,...,...,...
9764,TXN_1688292,UNKNOWN,3.0,NaN,9.0,Credit Card,In-store,NaN
9777,TXN_4385826,UNKNOWN,2.0,1.5,3.0,Credit Card,Takeaway,2023-08-02
9836,TXN_9162296,UNKNOWN,3.0,4.0,12.0,Cash,In-store,2023-05-10
9946,TXN_8807600,UNKNOWN,1.0,4.0,4.0,Cash,Takeaway,2023-09-24


In [6]:
data["item"].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

- Coffee Price : 2
- Cake Price : 3
- Cookie Price : 1
- Salad Price : 5
- Smoothie Price : 4
- Sandwich Price : 4
- Juice Price : 3
- Tea Price : 1.5

We can estimate the UNKNOWN item based on this price list, except for Smoothie and Sandwich (price = 4) and Cake and Juice (price = 3).

In [7]:
PRECISION = 2

price_to_item = {
    1.0: "Cookie",
    1.5: "Tea",
    2.0: "Coffee",
    5.0: "Salad"
}

mask = data["item"] == "UNKNOWN"

data.loc[mask, "item"] = (
    data.loc[mask, "price_per_unit"]
        .round(PRECISION)
        .map(price_to_item)
        .fillna("UNKNOWN")
)

data

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3.0,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3.0,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [8]:
data = data[data["item"] != "UNKNOWN"]
data

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3.0,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3.0,NaN,3.0,Digital Wallet,NaN,2023-12-02


## 4. Cleaning NaN Values From Numerical Variables

In [9]:
data = data.dropna(subset=["quantity"])
data = data.dropna(subset=["item"])
data = data.dropna(subset=["price_per_unit"])
data

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9992,TXN_2739140,Smoothie,4.0,4.0,16.0,UNKNOWN,In-store,2023-07-05
9993,TXN_4766549,Smoothie,2.0,4.0,NaN,Cash,NaN,2023-10-20
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,UNKNOWN,2023-08-30
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02


In [10]:
data["total_spent"] = data["quantity"] * data["price_per_unit"]
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8576 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    8576 non-null   object 
 1   item              8576 non-null   object 
 2   quantity          8576 non-null   float64
 3   price_per_unit    8576 non-null   float64
 4   total_spent       8576 non-null   float64
 5   payment_method    6376 non-null   object 
 6   location          5776 non-null   object 
 7   transaction_date  8445 non-null   object 
dtypes: float64(3), object(5)
memory usage: 603.0+ KB


## 5. Cleaning NaN Values From Object Variables

In [11]:
data["payment_method"].unique()

array(['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', nan, 'ERROR'],
      dtype=object)

In [12]:
data["location"].unique()

array(['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR'], dtype=object)

In [13]:
data = data[data["location"] != "UNKNOWN"]
data = data[data["location"] != "ERROR"]
data = data.dropna(subset=["location"])
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5173 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    5173 non-null   object 
 1   item              5173 non-null   object 
 2   quantity          5173 non-null   float64
 3   price_per_unit    5173 non-null   float64
 4   total_spent       5173 non-null   float64
 5   payment_method    3857 non-null   object 
 6   location          5173 non-null   object 
 7   transaction_date  5094 non-null   object 
dtypes: float64(3), object(5)
memory usage: 363.7+ KB


In [14]:
data = data[data["payment_method"] != "UNKNOWN"]
data = data[data["payment_method"] != "ERROR"]
data = data.dropna(subset=["payment_method"])
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3574 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    3574 non-null   object 
 1   item              3574 non-null   object 
 2   quantity          3574 non-null   float64
 3   price_per_unit    3574 non-null   float64
 4   total_spent       3574 non-null   float64
 5   payment_method    3574 non-null   object 
 6   location          3574 non-null   object 
 7   transaction_date  3521 non-null   object 
dtypes: float64(3), object(5)
memory usage: 251.3+ KB


In [15]:
data = data[data["transaction_date"] != "UNKNOWN"]
data = data[data["transaction_date"] != "ERROR"]
data = data.dropna(subset=["transaction_date"])
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3416 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    3416 non-null   object 
 1   item              3416 non-null   object 
 2   quantity          3416 non-null   float64
 3   price_per_unit    3416 non-null   float64
 4   total_spent       3416 non-null   float64
 5   payment_method    3416 non-null   object 
 6   location          3416 non-null   object 
 7   transaction_date  3416 non-null   object 
dtypes: float64(3), object(5)
memory usage: 240.2+ KB


At the end, we lost nearly 70% of the dataset. We can reconsider this approach by keeping NaN values in columns that are not essential to the business problem. Also, items with the same price but different names might not need to be removed and can be kept in the dataset.